# 08 — Fase 3: fine-tuning, assistente e LangGraph

**Tech Challenge Fase 3 — Assistente médico**

Este notebook concentra a demonstração da Fase 3 usando os dados que acompanham o projeto:

1. amostra anonimizada do SIVEP/OpenDataSUS;
2. protocolos oficiais de SRAG + PubMedQA + MedQuAD;
3. preparação do dataset de fine-tuning;
4. treino real de um Transformer pequeno em CPU para validar o pipeline;
5. assistente com RAG, contexto do paciente e guardrails;
6. fluxo de decisão do LangGraph;
7. avaliação e auditoria.

> O Transformer pequeno serve para validar o caminho de treino sem GPU. O fine-tuning do LLM pré-treinado por LoRA/PEFT continua em `src/finetuning/train_lora.py`.


## 1. Configuração

In [1]:
import os
import sys
from pathlib import Path

# O notebook pode ser aberto pela raiz do projeto ou pela pasta notebooks.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print("Raiz do projeto:", ROOT)


Raiz do projeto: /mnt/data/phase3_mr_work/colleague


## 2. Base estruturada de pacientes

O projeto inclui uma amostra com 8 mil registros reais do SIVEP, já sem campos de identificação pessoal. Nesta etapa ela é convertida para SQLite, que é o formato usado pelo `PatientDB`.

Os CSVs anuais completos também podem ser usados pelo mesmo adaptador sem mudar o restante do assistente.


In [2]:
from src.assistant import config
from src.assistant.knowledge.sivep_adapter import prepare_sqlite_from_prepared_csv
from src.assistant.knowledge.patient_db import PatientDB

# Recriamos o banco para garantir que o notebook não depende de um .db antigo.
quantidade = prepare_sqlite_from_prepared_csv(
    config.SIVEP_SAMPLE_PATH,
    config.SIVEP_DB_PATH,
)

db = PatientDB()
print(f"Pacientes carregados: {quantidade:,}")
print("Primeiro registro:")
print(db.get(db.todos_ids(limit=1)[0]))


Pacientes carregados: 8,000
Primeiro registro:
{'patient_id': 'P000118', 'ano_fonte': 2024, 'arquivo_fonte': 'INFLUD24-26-06-2025.csv', 'idade': 3.0, 'sexo': 'M', 'asma': 'não informado', 'cardiopati': 'não informado', 'desc_resp': 'sim', 'diabetes': 'não informado', 'diarreia': 'não', 'dispneia': 'sim', 'dor_abd': 'não', 'fadiga': 'não', 'fator_risc': 'não informado', 'febre': 'não', 'garganta': 'não', 'hepatica': 'não informado', 'hospital': 'sim', 'imunodepre': 'não informado', 'neurologic': 'não informado', 'nosocomial': 'não', 'obesidade': 'não informado', 'perd_olft': 'não', 'perd_pala': 'não', 'pneumopati': 'não informado', 'puerpera': 'não informado', 'renal': 'não informado', 'saturacao': 'sim', 'tosse': 'não', 'uti': 'não', 'vacina_cov': 'sim', 'vomito': 'não', 'suporte_ventilatorio': 'não invasivo', 'raio_x_torax': 'não realizado', 'tomografia': 'não realizado', 'tipo_amostra': 'secreção de naso-orofaringe', 'resultado_pcr': 'não detectável', 'classificacao_final': 'SRAG não

## 3. Anonimização textual

Além de não levar identificadores do SIVEP para o banco, o pipeline de fine-tuning possui uma etapa de limpeza para textos que eventualmente tragam nome, CPF, CNS, telefone ou e-mail.


In [3]:
from src.finetuning.dataset_prep import anonymize

exemplo = "Paciente João da Silva, CPF 123.456.789-00, email joao@x.com"
print("Antes :", exemplo)
print("Depois:", anonymize(exemplo))


Antes : Paciente João da Silva, CPF 123.456.789-00, email joao@x.com
Depois: [NOME_REMOVIDO], CPF [CPF_REMOVIDO], email [EMAIL_REMOVIDO]


## 4. Dataset de fine-tuning

A preparação junta quatro grupos:

- MedQuAD;
- PubMedQA;
- perguntas curadas dos protocolos oficiais de SRAG;
- poucos exemplos sintéticos de formatos internos, como laudo/receita/procedimento.

Os exemplos passam por limpeza, anonimização, deduplicação e split determinístico.


In [4]:
from src.finetuning.dataset_prep import build_dataset

stats = build_dataset()
print("Total único:", stats["total_exemplos_unicos"])
print("Treino:", stats["train_linhas_gravadas"])
print("Validação:", stats["val"])
print("Teste:", stats["test"])
print()
print("Por fonte:")
for fonte, total in stats["por_fonte"].items():
    print(f"- {fonte}: {total}")


Total único: 2587
Treino: 2095
Validação: 258
Teste: 258

Por fonte:
- MedQuAD: 1500
- PubMedQA: 1000
- Protocolos oficiais SRAG: 100
- Protocolos internos (sintéticos): 6


### 4.1 Conferência do caminho LoRA

O treino final com uma LLM pré-treinada usa LoRA/PEFT. O `dry-run` verifica se o dataset está no formato esperado sem baixar o modelo — útil antes de mandar o job para uma GPU.


In [5]:
import subprocess

resultado = subprocess.run(
    [sys.executable, "-m", "src.finetuning.train_lora", "--dry-run"],
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(resultado.stdout)


[train_lora] modelo base: tiiuae/falcon-7b-instruct
[train_lora] treino: /mnt/data/phase3_mr_work/colleague/data/finetuning/processed/train.jsonl | validação: /mnt/data/phase3_mr_work/colleague/data/finetuning/processed/val.jsonl
[train_lora] dry-run OK — 2095 exemplos de treino no formato esperado.



## 5. Validação de treino real em CPU

Para não depender de uma curva de loss simulada, esta etapa treina de verdade um Transformer pequeno com os protocolos e as perguntas de SRAG. Ele não substitui Falcon/LLaMA/Mistral; serve para provar que preparação, treino, checkpoint e inferência funcionam de ponta a ponta em uma máquina comum.


In [6]:
from src.finetuning.local_validation import run as treinar_local

metricas_treino = treinar_local()
for chave in (
    "pretraining_pairs",
    "training_examples",
    "evaluation_examples",
    "pretraining_final_loss",
    "finetuning_final_loss",
    "mean_token_f1",
):
    print(f"{chave}: {metricas_treino[chave]}")


/opt/pyvenv/lib/python3.13/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


pretraining_pairs: 95
training_examples: 80
evaluation_examples: 20
pretraining_final_loss: 5.782823244730632
finetuning_final_loss: 3.4401249170303343
mean_token_f1: 0.19406178416986414


## 6. RAG e fontes

O retriever pesquisa ao mesmo tempo nos protocolos oficiais, nos documentos originais do projeto, no MedQuAD e no PubMedQA. Para os documentos oficiais, a referência mantém arquivo e página.


In [7]:
from src.assistant.knowledge.retriever import ProtocolRetriever

retriever = ProtocolRetriever()
print(f"Trechos indexados: {retriever.total_trechos():,}")

for trecho in retriever.buscar("Quais sinais indicam SRAG?", top_k=3):
    print(f"- {trecho.citacao()} | score={trecho.score:.3f}")


Trechos indexados: 1,471
- guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 27 | score=0.087
- guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 54 | score=0.077
- PROT-SRAG-01 — Sinais de alerta para reavaliação imediata | score=0.071


## 7. Assistente médico contextualizado

O assistente consulta o paciente no SQLite, recupera os trechos mais próximos e passa o contexto para o backend disponível. Quando o checkpoint local existe, ele é usado automaticamente como validação do modelo ajustado.


In [8]:
from src.assistant.chains.medical_assistant import MedicalAssistant

assistant = MedicalAssistant()
print("Backend:", assistant.llm.backend)

patient_id = db.primeiro_por_risco("vermelho") or db.todos_ids(limit=1)[0]
print("Paciente escolhido:", patient_id)
print(db.resumo_clinico(patient_id))

resposta = assistant.responder(
    "Quais sinais deste caso merecem atenção segundo os protocolos?",
    paciente_id=patient_id,
)
print()
print("RESPOSTA:")
print(resposta.resposta)
print()
print("FONTES:")
for fonte in resposta.fontes:
    print("-", fonte)


Backend: finetuned-local
Paciente escolhido: P387726
Paciente P387726; idade: 5.0; sexo: M; febre: sim; UTI: sim; suporte ventilatório: não utilizou; PCR: detectável; classificação final: SRAG por outro vírus respiratório; desfecho registrado: cura; ano do registro: 2023; risco derivado para o fluxo: vermelho; exames pendentes: nenhum registrado como pendente.



RESPOSTA:
o guia orienta de registra, módulo, como, como, como, como, como, como, como, como, como, como, como, como, como, como, como, como, com, como, não e, não e, não e, não e, não.

⚠️ Este conteúdo é apoio à consulta dos protocolos. Condutas, prescrições e doses precisam de avaliação e validação do médico responsável.

Fontes consultadas: guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 64; guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 50; guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 69; guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 30.

FONTES:
- guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 64
- guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 50
- guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 69
- guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 30


### 7.1 Guardrail de prescrição

O bloqueio acontece antes da geração e a saída também é revisada. Assim, uma frase com dose imperativa não é mantida só porque existe um aviso médico no final.


In [9]:
pedido = assistant.responder(
    "Prescreva a dose exata em mg de corticoide para este paciente",
    paciente_id=patient_id,
)
print("Bloqueado:", pedido.bloqueado_guardrail)
print(pedido.resposta)


Bloqueado: True
Não posso definir prescrição, dose ou posologia diretamente. Posso localizar os critérios e orientações descritos nos protocolos, mas a decisão terapêutica precisa ser feita e validada pelo médico responsável.

Fontes consultadas: guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 31; guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 147; guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 96; guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 109.


## 8. Fluxo de decisão com LangGraph

Com `langgraph` instalado, a classe compila o fluxo em `StateGraph`. No ambiente de teste sem a dependência, os mesmos nós rodam em Python puro, o que permite validar as regras sem manter duas lógicas diferentes.


In [10]:
from src.assistant.chains.graph import FluxoAtendimento

fluxo = FluxoAtendimento(assistant=assistant)
estado = fluxo.executar(patient_id, "Faça a triagem deste caso")

print("Trilha:", " -> ".join(estado["trilha"]))
print()
print("Resumo final:")
print(estado["resumo_final"])


Trilha: triagem -> verificar_exames -> emitir_alerta -> consolidar

Resumo final:
Paciente P387726 | risco: vermelho

Exames pendentes: nenhum registrado

🚨 ALERTA — paciente P387726 classificado como risco vermelho pelos indicadores disponíveis na base. Priorizar avaliação da equipe médica. Não há exame registrado como pendente.

Orientação baseada nas fontes:
o guia orienta a coleta e de início, como, mas e srag.

⚠️ Este conteúdo é apoio à consulta dos protocolos. Condutas, prescrições e doses precisam de avaliação e validação do médico responsável.

Fontes consultadas: guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 50; guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 30; guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 41; guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 63.

Fontes: guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 50; guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 30; guia_ms_vigilan

## 9. Avaliação

As perguntas de avaliação ficam separadas da curadoria usada no treino. Aqui verificamos recuperação da fonte/página, presença do aviso médico, bloqueio de prescrição e dois testes de sanidade para MedQuAD/PubMedQA.


In [11]:
from src.assistant.finetuning.evaluate import avaliar

resultado_avaliacao = avaliar(assistant)
for chave, valor in resultado_avaliacao.items():
    if isinstance(valor, float):
        print(f"{chave}: {valor:.2%}")
    else:
        print(f"{chave}: {valor}")


n_perguntas_gold: 20
source_recall_at_k: 80.00%
source_page_recall_at_k: 70.00%
taxa_disclaimer: 100.00%
taxa_bloqueio_prescricao: 100.00%
backend: finetuned-local
composicao_dataset_finetuning: {'faq': 10, 'modelo_laudo': 1, 'modelo_receita': 1, 'modelo_procedimento': 1, 'protocolo': 14, 'protocolo_oficial': 100, 'pubmedqa': 250, 'medquad': 350}
retrieval_hit_medquad: 100.00%
retrieval_hit_pubmedqa: 100.00%
detalhes: [{'pergunta': 'O que caracteriza uma síndrome gripal suspeita de covid-19?', 'fonte_esperada': 'guia_srag_ministerio_saude_2025.pdf', 'pagina_esperada': 4, 'fontes_recuperadas': ['guia_srag_ministerio_saude_2025.pdf, p. 3', 'guia_srag_ministerio_saude_2025.pdf, p. 4', 'guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 33', 'guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 122', 'guia_srag_ministerio_saude_2025.pdf, p. 2'], 'fonte_ok': True, 'pagina_ok': True}, {'pergunta': 'O que fazer quando um paciente com síndrome gripal apresenta fatores de ri

## 10. Auditoria

Cada consulta registra informações suficientes para rastrear o que aconteceu: pergunta, paciente, backend, fontes recuperadas e decisões dos guardrails.


In [12]:
from src.assistant.safety.audit_log import AuditLogger

logs = AuditLogger().ler_eventos()
print(f"Eventos registrados: {len(logs)}")
if logs:
    ultimo = logs[-1]
    print("Último evento:")
    print(ultimo)


Eventos registrados: 34
Último evento:
{'pergunta': '[fluxo] risco vermelho', 'resposta': '🚨 ALERTA — paciente P387726 classificado como risco vermelho pelos indicadores disponíveis na base. Priorizar avaliação da equipe médica. Não há exame registrado como pendente.', 'backend_llm': 'finetuned-local', 'paciente_id': 'P387726', 'fontes': ['guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 50', 'guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 30', 'guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 41', 'guia_ms_vigilancia_integrada_virus_respiratorios_2024.pdf, p. 63'], 'guardrail_bloqueou': False, 'guardrail_categorias': [], 'fluxo_no': 'emitir_alerta', 'timestamp': '2026-08-15T22:29:10.703584+00:00'}


---
## Conclusão

A demonstração usa dados reais anonimizados do SIVEP para contexto estruturado, protocolos oficiais com página, PubMedQA/MedQuAD para ampliar a cobertura, treino real em CPU para validar o pipeline e o caminho LoRA/PEFT para o ajuste da LLM pré-treinada.

O fluxo mantém as exigências de segurança: pedidos de prescrição são bloqueados, respostas trazem fontes e toda conduta precisa de validação médica.
